In [9]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from pathlib import Path
import os

In [ ]:
# Define the path to the dataset (Note: 0 is real, while 1 is fake)
test_path = Path("Enter path to test dataset here")
train_path = Path("Enter path to train dataset here")


# Transforms
train_transformer = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

test_transformer = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)
print(f"Train path: {train_path}, Test path: {test_path}")
# Conversion of files to PyTorch Dataset and DataLoader
train_data = datasets.ImageFolder(root=train_path, transform=train_transformer)
test_data = datasets.ImageFolder(root=test_path, transform=test_transformer)

train_loader = DataLoader(train_data, batch_size=50, shuffle=True, num_workers=4)
test_loader = DataLoader(test_data, batch_size=50, shuffle=False, num_workers=4)

Train path: D:\FiveKDataset\GAN\Train, Test path: D:\FiveKDataset\GAN\Test


In [11]:
# Create a binary classification ResNet50 model
def BinaryResnet50(pretrained=True, freeze_features=True):
    # Load pretrained ResNet50
    model = models.resnet50(weights="DEFAULT" if pretrained else None)

    # Freeze layers if True
    if freeze_features:
        for param in model.parameters():
            param.requires_grad = False

    # Replace the final fully connected layer
    model.fc = nn.Linear(2048, 1)

    return model


model = BinaryResnet50(pretrained=True, freeze_features=False)
model = model.to("cuda")

# print(model)

In [12]:
# Define loss , optimizer, and learning rate scheduler, accuracy, false pos, false neg functions
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0002)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.1, patience=3)

In [16]:
# Training Loop
train_size = 10000
acc_list = []
fp_list = []
fn_list = []
loss_list = []

test_size = 1000
test_loss_list = []
test_acc_list = []
test_fp_list = []
test_fn_list = []

epoch_list = [i for i in range(1, 101)]
EPOCHS = 1

for epoch in range(EPOCHS):
    # Model Training
    model.train()
    
    correct = 0
    fp = 0
    fn = 0
    running_loss = 0
    losses = []
    
    
    for i, inp in enumerate(train_loader):
        inputs, labels = inp
        inputs, labels = inputs.to("cuda"), labels.to("cuda")

        outputs = model(inputs).squeeze()
        preds = torch.round(torch.sigmoid(outputs))
        
        
        loss = loss_fn(outputs, labels.type(torch.float))
        losses.append(loss.item())
        correct += (preds.view(-1) == labels).sum().item()
        fp += ((preds.view(-1) == 1) & (labels == 0)).sum().item()
        fn += ((preds.view(-1) == 0) & (labels == 1)).sum().item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        print(f"Batch {i + 1}, Loss: {loss.item()}")

    acc_list.append(correct / train_size * 100)
    fp_list.append(fp / train_size  * 100)
    fn_list.append(fn / train_size * 100)
    avg_loss = sum(losses) / len(losses)
    loss_list.append(avg_loss)
    scheduler.step(avg_loss)
    
    # Model Evaluation
    model.eval()
    
    correct = 0
    fp = 0
    fn = 0
    running_loss = 0
    losses = []
    
    with torch.inference_mode():
        for i, inp in enumerate(test_loader):
            inputs, labels = inp
            inputs, labels = inputs.to("cuda"), labels.to("cuda")

            outputs = model(inputs).squeeze()
            preds = torch.round(torch.sigmoid(outputs))
            
            
            loss = loss_fn(outputs, labels.type(torch.float))
            losses.append(loss.item())
            correct += (preds.view(-1) == labels).sum().item()
            fp += ((preds.view(-1) == 1) & (labels == 0)).sum().item()
            fn += ((preds.view(-1) == 0) & (labels == 1)).sum().item()

            running_loss += loss.item()

    avg_loss = sum(losses) / len(losses)
    test_loss_list.append(avg_loss)
    test_acc_list.append(correct / test_size * 100)
    test_fp_list.append(fp / test_size * 100)
    test_fn_list.append(fn / test_size * 100)

    print(f"Epoch {epoch + 1}, Train Loss: {loss_list[epoch]}, Train Acc: {acc_list[epoch]}%, FP: {fp_list[epoch]}%, FN: {fn_list[epoch]}%")
    print(f"Epoch {epoch + 1}, Test Loss: {test_loss_list[epoch]}, Test Acc: {test_acc_list[epoch]}%, FP: {test_fp_list[epoch]}%, FN: {test_fn_list[epoch]}%")

    

Batch 1, Loss: 0.0017775475280359387
Batch 2, Loss: 0.0035859234631061554
Batch 3, Loss: 0.02560574561357498
Batch 4, Loss: 0.0008522443822585046
Batch 5, Loss: 0.0014961491106078029
Batch 6, Loss: 0.0013613224728032947
Batch 7, Loss: 0.014781718142330647
Batch 8, Loss: 0.01979791559278965
Batch 9, Loss: 0.0009236750775016844
Batch 10, Loss: 0.005330829415470362
Batch 11, Loss: 0.0037258844822645187
Batch 12, Loss: 0.0011884388513863087
Batch 13, Loss: 0.00868083257228136
Batch 14, Loss: 0.008829980157315731
Batch 15, Loss: 0.0013008364476263523
Batch 16, Loss: 0.00254120584577322
Batch 17, Loss: 0.0036785125266760588
Batch 18, Loss: 0.01660284772515297
Batch 19, Loss: 0.038918636739254
Batch 20, Loss: 0.02775174006819725
Batch 21, Loss: 0.0013008539099246264
Batch 22, Loss: 0.0012475567637011409
Batch 23, Loss: 0.0550997257232666
Batch 24, Loss: 0.00036090525100007653
Batch 25, Loss: 0.011195349507033825
Batch 26, Loss: 0.005522640887647867
Batch 27, Loss: 0.002587781986221671
Batch 2